# 04 · Kidney segmentation (Phase 2 of `plan.md`)

Runs the Phase 2 pipeline: S-A U-Net ResNet-34 [S18][S17], S-B DeepLabV3+ ([24] Table 5), S-C MedSAM [S3] and S-D UltraSam [S1] with box prompts, the optional MedSAM decoder fine-tune, the external test on KidneyUS [18], and the results table against URI-CADS [24] (Dice 91.23 %, IoU 84.99 %).

All logic lives in `src/kidney/`; this notebook only calls it, so the same code runs from a terminal (`bash scripts/run_segmentation.sh`).

**Set `SMOKE = True`** to check every step on CPU in a few minutes (tiny subset, 1 epoch, random weights). Numbers from a smoke run mean nothing.

In [ ]:
SMOKE = False          # True = quick CPU check, False = real run on the GPU
BRANCH = "segmentation"
REPO_URL = "https://github.com/Noadori34/Project-data-science.git"
REPO_DIR = "/content/Project-data-science"   # Colab; change for the lab machine

In [ ]:
import os, sys, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!pip -q install -r requirements-seg.txt
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
os.environ["PYTHONPATH"] = os.path.join(REPO_DIR, "src")
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
S = "--smoke" if SMOKE else ""

## 1. Shared protocol: tests and the 5-fold split (plan 1.2)
The team's leak-free split (`results/eda/folds.csv`, near-duplicates kept in one fold) becomes `results/splits/folds.csv`. Repetition *k*: test = fold *k*, validation = fold *k*+1, train = the other three.

In [ ]:
!python -m pytest tests -q
!python -m kidney.splits

## 2. Masks: `.mat` → PNG and checks (plan 3.1)

In [ ]:
!python -m kidney.prepare_masks
import pandas as pd
chk = pd.read_csv("results/segmentation/mask_check.csv")
chk.describe()[["area_frac", "n_components", "hole_px"]]

## 3. S-A and S-B under 5-fold CV (plan 3.3)
Dice loss [S24] + BCE, AdamW [S30], cosine schedule, early stopping on validation Dice, largest region + hole filling. Each fold takes roughly 10–20 min on one GPU.

In [ ]:
!python -m kidney.train_seg --model unet_r34 {S}

In [ ]:
!python -m kidney.train_seg --model deeplabv3p_r50 {S}

## 4. S-C MedSAM and S-D UltraSam with box prompts (plan 3.4)
`gt` = box from the true mask (upper bound); `auto` = box from the S-A out-of-fold mask (fully automatic). Needs the weights in `weights/`.

In [ ]:
if not SMOKE:
    !bash scripts/download_weights.sh
!python -m kidney.sam_prompt --model medsam {S}
!python -m kidney.sam_prompt --model ultrasam {S}

### Optional: fine-tune the MedSAM mask decoder on the training folds [S3]

In [ ]:
!python -m kidney.finetune_medsam {S}

## 5. External test on KidneyUS [18] (plan 3.5)
Dice against the capsule and against an ellipse fitted to the capsule; split by view, quality and native vs transplant.

In [ ]:
!python -m kidney.eval_external --models unet_r34 deeplabv3p_r50 medsam ultrasam medsam_ft {S}
ext = pd.read_csv("results/segmentation/external/external_summary.csv")
ext[ext.member.isin(["ens", "gt", "auto"])].round(4)

## 6. Results table, `kidney_boxes.csv`, failure cases

In [ ]:
!python -m kidney.report_seg
pd.read_csv("results/segmentation/summary.csv")

In [ ]:
from IPython.display import Image, display
import glob
for g in sorted(glob.glob("results/segmentation/failures/*_grid.png")):
    print(g); display(Image(g, width=900))

## 7. Save results to the repo
Checkpoints, caches and weights are git-ignored; per-image CSVs, predicted masks, tables and overlays are committed.

In [ ]:
# !git add results/splits results/segmentation results/pred_masks results/runs.csv results/kidney_boxes.csv
# !git commit -m "Phase 2 segmentation results" && git push origin segmentation